```mermaid
flowchart LR
    A["参数 Tensor<br/>requires_grad=True"] --> B["Forward"]
    B --> C["动态构建 Autograd DAG"]
    C --> D["loss"]
    D --> E["loss.backward()"]
    E --> F["按链式法则反向传播"]
    F --> G["leaf 参数的 .grad"]
    G --> H["optimizer.step()"]
    H --> I["optimizer.zero_grad()"]
    I --> B
```

> requires_grad 决定是否需要参与梯度追踪；grad_fn 描述 Tensor 是由什么运算产生的；.grad 保存最终累积下来的梯度。

# 从 requires_grad 到动态计算图

requires_grad 是 Tensor 上的布尔属性，用来告诉 autograd：从这个 Tensor 出发的相关可微运算是否需要建立自动微分历史。常见入口有创建 Tensor 时传入 requires_grad=True，以及之后调用原地状态修改方法 .requires_grad_(True)。PyTorch 官方文档将 .requires_grad_() 的主要用途描述为“开始让 autograd 记录该 Tensor 上的运算”。Autograd 目前用于浮点与复数 Tensor，而不是普通整数 Tensor

In [5]:
import torch

# 方法一：创建时开启梯度追踪
x = torch.tensor(2.0, requires_grad=True)

# 方法二：已有 Tensor 后开启
w = torch.tensor(3.0)
w.requires_grad_(True)

print(x.requires_grad)  # True
print(w.requires_grad)  # True

y = x * w + 1
print(y.requires_grad)  # True

True
True
True


PyTorch 的动态计算图（Autograd Engine）遵循一条底层法则：梯度的需求状态在正向传播中自底向上“传染”，而模型的冻结只需从根节点“切断”。

理解这一机制需要从以下四个维度层层递进。

梯度传染规则：有向无环图（DAG）中的“逻辑或”

在普通 eager-mode 下，对于任意可微算子 $y = f(x_1, x_2, \dots, x_n)$：

传播逻辑：只要输入中至少有一个浮点张量的 requires_grad == True，输出 $y$ 的 requires_grad 就必定自动为 True。

底层机制：PyTorch 会为 $y$ 挂载一个负责反向传播求导的节点 grad_fn（例如 AddBackward0 或 MmBackward0）。

必然性：链式法则要求 $\frac{\partial \mathcal{L}}{\partial x_i} = \frac{\partial \mathcal{L}}{\partial y} \cdot \frac{\partial y}{\partial x_i}$。只要上游任何一个输入 $x_i$ 未来需要梯度，计算图就必须记录从 $y$ 到 $x_i$ 的导数求值路径，否则梯度流动会在 $y$ 处中断。

In [6]:
x = torch.randn(2, 2, requires_grad=False)
w = torch.randn(2, 2, requires_grad=True)

y = x @ w
print(y.requires_grad)  # 输出: True
print(y.grad_fn)        # 输出: <MmBackward0 ...>

True


上下文管理器的绝对覆写

torch.no_grad() 与 torch.inference_mode() 并不是去修改输入 Tensor 的属性，而是操作了线程局部的全局开关（C++ 层的 GradMode::is_enabled()）：

- 覆写传染：即使输入的所有张量 requires_grad=True，在 with torch.no_grad(): 代码块内执行运算时，调度器会直接跳过计算图的构建逻辑。
- 结果状态：生成的新 Tensor 其 requires_grad 强制为 False，且 grad_fn 为 None。
- 显存与算力收益：算子内部不会执行 ctx.save_for_backward(...)，不需要缓存前向激活值，大幅降低推理与评估时的显存开销。

Leaf 与 Non-Leaf Tensor 的本质差异

PyTorch 计算图严格区分两类节点：

Leaf Tensor（叶子节点）：用户直接初始化的源头张量（如 nn.Parameter、输入的 batch 数据）。其 grad_fn 为 None，是梯度的终点（累积存储在 .grad 中）。

Non-Leaf Tensor（非叶子节点）：由操作生成的中间产物（如每层的激活值、中间损失值）。其 requires_grad 完全由父节点的属性和计算历史自动推导。

In [7]:
import torch.nn as nn

model = nn.Linear(4, 2)

for name, p in model.named_parameters():
    print(name, p.requires_grad)

weight True
bias True


nn.Parameter 被注册为模型参数时默认是可训练参数，这也是为什么普通神经网络参数通常天然带有 requires_grad=True。

In [8]:
# x = torch.tensor([1, 2, 3], requires_grad=True)

整数 Tensor 并不是普通 reverse-mode autograd 的目标类型；需要梯度的连续优化变量通常应使用浮点或复数 dtype

另一个错误是认为：所有 Tensor 都应该 requires_grad=True
实际上训练数据、标签等通常不需要参数梯度

```python
x = torch.randn(32, 10)       # 输入通常不需要求梯度
target = torch.randn(32, 1)

model = nn.Linear(10, 1)      # 参数需要梯度
pred = model(x)
```

In [9]:
x = torch.tensor(2.0, requires_grad=True)
a = x * 3
b = torch.tensor(5.0)
c = a + b

print(x.requires_grad)
print(a.requires_grad)
print(b.requires_grad)
print(c.requires_grad)

True
True
False
True


为什么 b 不需要梯度，却不妨碍 c 对 x 求导？

$$ c=3x+5,\qquad \frac{\partial c}{\partial x}=3$$

常数 5 自己不需要成为优化变量。

```python
for p in model.parameters():
    p.requires_grad_(False)

# 仅重新训练某个层
for p in model.classifier.parameters():
    p.requires_grad_(True)
```

**leaf / non-leaf Tensor**

这是 Autograd 最容易误解的概念之一。

对于 requires_grad=True 的 Tensor，由用户直接创建、不是某个被记录运算结果的 Tensor 通常是 leaf；通过可微运算得到的结果通常是 non-leaf。 对 requires_grad=False 的 Tensor，PyTorch 按约定把它们也视为 leaf。

In [10]:
x = torch.tensor(2.0, requires_grad=True)
y = x * 3
z = y ** 2

print(x.is_leaf)  # True
print(y.is_leaf)  # False
print(z.is_leaf)  # False

True
False
False


为你转换的 Markdown 表格如下：

| Tensor 类型 | `requires_grad` | `is_leaf` | `grad_fn` | backward 后 `.grad` 默认保存？ |
| --- | --- | --- | --- | --- |
| 可训练模型参数 | `True` | `True` | `None` | 是 |
| 用户创建的可训练 Tensor | `True` | `True` | `None` | 是 |
| 中间计算结果 | `True` | `False` | 有 | 否 |
| 普通数据 Tensor | `False` | `True`（约定） | `None` | 否 |
| `detach()` 结果 | `False` | 通常为 leaf | `None` | 否 |

在 PyTorch 的计算图中，叶子节点（Leaf Tensor）是整个计算链条的“源头起点”，而非叶子节点（Non-Leaf Tensor）则是运算过程中产生的“中间变量”。

你可以把前向传播想象成烹饪过程：你直接买来的原始食材是叶子节点，而切好的肉丝、热好的底汤这些中间加工步骤则是非叶子节点。

如何判断一个 Tensor 是不是叶子节点？

PyTorch 的判定规则其实非常简单直接：

叶子节点 (is_leaf = True)：

- 由用户亲手直接创建的 Tensor（只要不是算出来的）：比如 torch.randn(...)、torch.tensor(...) 以及神经网络的权重参数 nn.Parameter。
- 所有 requires_grad=False 的 Tensor：既然不需要求导，系统就没有必要记录它的计算历史，默认全都视为叶子节点。

非叶子节点 (is_leaf = False)：

- 由其他 Tensor 计算得到的中间结果（且其输入中有需要梯度的变量）：比如 $y = w \cdot x + b$ 中的 $y$。$y$ 携带了生成它的算子记录（grad_fn），因此是中间派生出来的非叶子节点。

为什么 PyTorch 要特意区分它们？

核心目的只有一个：极大地节省显存（内存）。

在深度学习反向传播（loss.backward()）时，我们的最终目标是更新模型参数（权重、偏置等叶子节点）。

In [13]:
import torch

# 1. 用户直接初始化的权重（起点）
w = torch.tensor([2.0], requires_grad=True)
b = torch.tensor([1.0], requires_grad=True)

# 2. 纯数据输入，不需要求导
x = torch.tensor([3.0], requires_grad=False)

# 3. 计算出的中间变量
y = w * x + b

print(w.is_leaf)  # True: 用户直接创建的权重，是叶子
print(x.is_leaf)  # True: requires_grad=False 的数据，约定为叶子
print(y.is_leaf)  # False: 通过加法乘法算出来的中间产物，是非叶子

True
True
False


In [14]:
x = torch.tensor(2.0, requires_grad=True)
y = x * 3
z = y ** 2

z.backward()

print(x.grad)
print(y.grad)

tensor(36.)
None


C:\Users\13127\AppData\Local\Temp\ipykernel_34696\1298782211.py:8: UserWarning: The .grad attribute of a Tensor that is not a leaf Tensor is being accessed. Its .grad attribute won't be populated during autograd.backward(). If you indeed want the .grad field to be populated for a non-leaf Tensor, use .retain_grad() on the non-leaf Tensor. If you access the non-leaf Tensor by mistake, make sure you access the leaf Tensor instead. See github.com/pytorch/pytorch/pull/30531 for more information. (Triggered internally at C:\actions-runner\_work\pytorch\pytorch\build\aten\src\ATen/core/TensorBody.h:493.)
  print(y.grad)


$$z=(3x)^2=9x^2$$
$$\frac{dz}{dx}=18x$$

在 x=2 时：$\frac{dz}{dx}=36 $

y.grad is None 并不意味着 backward 失败。Autograd 在反向传播过程中当然计算了通过 y 所需的中间梯度，但默认不会把 non-leaf Tensor 的梯度长期存进 .grad；如果确实要查看，需要调用 retain_grad()。官方教程特别演示了这一行为

**grad_fn**

grad_fn 不是 gradient。

> “为了反向传播，这个 Tensor 保存着指向生成它的 autograd Function 的入口。”

Tensor 运算对应 autograd Function 节点；输出 Tensor 的 grad_fn 可以用于访问其梯度计算历史，而 next_functions 还能继续沿图向前追踪。

In [16]:
x = torch.tensor(2.0, requires_grad=True)

a = x * 3
b = a ** 2
loss = b + 1

print(x.grad_fn)
print(a.grad_fn)
print(b.grad_fn)
print(loss.grad_fn)

print(loss.grad_fn.next_functions)

None
((<PowBackward0 object at 0x00000201A293BAF0>, 0), (None, 0))


grad_fn（Gradient Function）是 Tensor 的运算溯源指针，它记录了具体是哪一个算子（Operation）生成了当前这个 Tensor。当调用 backward() 时，Autograd 引擎正是顺着各个 Tensor 的 grad_fn 逆向回溯，一步步执行链式法则求导。

**核心机制：计算图的反向导航链**

在前向传播（Forward）中，每次对需要梯度的 Tensor 进行数学运算，PyTorch 都会在后台动态构建一张有向无环图（DAG）：

- 叶子节点（起点）：用户直接初始化的 Tensor 或模型权重，不是通过任何算子计算出来的，因此它们的 grad_fn 永远是 None。
- 非叶子节点（中间产物）：通过算子计算生成。PyTorch 会把该算子对应的反向导数计算逻辑封装成一个 Node 对象，挂载到该 Tensor 的 grad_fn 上。
- 链式追踪：每个 grad_fn 内部都有一个元组属性 next_functions，记录了输入该算子的上游节点的反向指针。这就是反向传播能够沿着整条计算路径“顺藤摸瓜”的秘密。

In [17]:
import torch

# 1. 初始化叶子节点
x = torch.tensor(2.0, requires_grad=True)
w = torch.tensor(3.0, requires_grad=True)

# 2. 乘法运算
y = x * w

# 3. 加法运算
z = y + 5.0

print("x.grad_fn:", x.grad_fn)  # 输出: None（叶子节点没有 grad_fn）
print("y.grad_fn:", y.grad_fn)  # 输出: <MulBackward0 object at 0x...>
print("z.grad_fn:", z.grad_fn)  # 输出: <AddBackward0 object at 0x...>

# 4. 窥探反向调用链（next_functions）
print("z 的上游来源:", z.grad_fn.next_functions)
# 输出结构形如: ((<MulBackward0 object at 0x...>, 0), (None, 0))
# 含义：z 的第 0 个输入来自 y (MulBackward0)，第 1 个输入是常数 5.0（不需要求导，故为 None）

x.grad_fn: None
y.grad_fn: <MulBackward0 object at 0x00000201A298B5B0>
z.grad_fn: <AddBackward0 object at 0x00000201A291FD90>
z 的上游来源: ((<MulBackward0 object at 0x00000201A298B5B0>, 0), (None, 0))


**常见的 grad_fn 类型**

grad_fn 的名字直观对应了正向执行的操作加上 Backward 后缀：

- <AddBackward0>：执行了加法运算（如 +）。
- <MulBackward0>：执行了乘法运算（如 *）。
- <MmBackward0>：执行了二维矩阵乘法（torch.mm）。
- <ReluBackward0>：执行了激活函数 ReLU 计算。
- <AccumulateGrad>：虽然叶子节点的 .grad_fn 显示为 None，但在计算图内部，上游中间节点的 next_functions 会最终指向一个叫 AccumulateGrad 的特殊节点，它专门负责把流回到叶子节点的梯度累加写入到叶子的 .grad 属性中。

**动态计算图 / DAG**

Autograd 可以把一次 forward 看成一张有向无环图：

- Tensor/操作构成计算依赖；
- edges 表示数据或梯度传播关系；
- forward 沿输入到输出执行；
- backward 沿输出依赖反向应用局部导数

In [23]:
x = torch.tensor(2.0, requires_grad=True)
a = 3 * x
b = a ** 2
loss = b + 1

![img3](./src/img3.png)